In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn.csv')

df['TotalCharges'] = df['TotalCharges'].replace(" ", np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
df['T_MODE'] = df['TotalCharges'].fillna(df['TotalCharges'].mode()[0])
df = df.drop('TotalCharges', axis=1)

df['Churn'] = df['Churn'].map({'Yes':1, 'No':0}).astype(int)
df = df.drop(['customerID'],axis=1)

In [ ]:
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Churn,T_MODE
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,0,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,0,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,1,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,0,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,1,151.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,0,1990.50
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,0,7362.90
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,0,346.45
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,1,306.60


In [ ]:
# Add SIM column using your mapping logic
def assign_indian_operator(row):
    if row['InternetService'] == 'Fiber optic':
        return 'Reliance Jio'
    elif row['InternetService'] == 'DSL':
        return 'Airtel'
    elif row['PhoneService'] == 'Yes' and row['MultipleLines'] == 'Yes':
        return 'Vodafone Idea'
    else:
        return 'BSNL'

df['Sim'] = df.apply(assign_indian_operator, axis=1)

In [ ]:

from sklearn.model_selection import train_test_split

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=45
)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [ ]:
X_train_nums = X_train.select_dtypes(exclude=object)
X_train_cats = X_train.select_dtypes(include=object)

X_test_nums = X_test.select_dtypes(exclude=object)
X_test_cats = X_test.select_dtypes(include=object)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(drop='first')

simple_cols = ['gender','Partner','Dependents']

ohe.fit(X_train_cats[simple_cols])

train_ohe = pd.DataFrame(ohe.transform(X_train_cats[simple_cols]).toarray(),
                         columns=ohe.get_feature_names_out())

test_ohe = pd.DataFrame(ohe.transform(X_test_cats[simple_cols]).toarray(),
                        columns=ohe.get_feature_names_out())

train_ohe.reset_index(drop=True, inplace=True)
test_ohe.reset_index(drop=True, inplace=True)

X_train_cats = pd.concat([X_train_cats, train_ohe], axis=1)
X_test_cats = pd.concat([X_test_cats, test_ohe], axis=1)

X_train_cats = X_train_cats.drop(simple_cols, axis=1)
X_test_cats = X_test_cats.drop(simple_cols, axis=1)

In [ ]:

from sklearn.preprocessing import OrdinalEncoder

ord_cols = ['PhoneService','MultipleLines','InternetService','OnlineSecurity',
            'OnlineBackup','DeviceProtection','TechSupport','StreamingTV',
            'StreamingMovies','Contract','PaperlessBilling','PaymentMethod','Sim']

oe = OrdinalEncoder()
oe.fit(X_train_cats[ord_cols])

train_ord = pd.DataFrame(oe.transform(X_train_cats[ord_cols]),
                         columns=[col+'_OD' for col in ord_cols])

test_ord = pd.DataFrame(oe.transform(X_test_cats[ord_cols]),
                        columns=[col+'_OD' for col in ord_cols])

train_ord.reset_index(drop=True,inplace=True)
test_ord.reset_index(drop=True,inplace=True)

X_train_cats = pd.concat([X_train_cats, train_ord], axis=1)
X_test_cats = pd.concat([X_test_cats, test_ord], axis=1)

X_train_cats = X_train_cats.drop(ord_cols, axis=1)
X_test_cats = X_test_cats.drop(ord_cols, axis=1)

X_train_nums = X_train_nums.reset_index(drop=True)
X_train_cats = X_train_cats.reset_index(drop=True)

In [ ]:


Training_data = pd.concat([X_train_nums, X_train_cats], axis=1)

X_test_nums = X_test_nums.reset_index(drop=True)
X_test_cats = X_test_cats.reset_index(drop=True)

Testing_data = pd.concat([X_test_nums, X_test_cats], axis=1)

print("FINAL TRAIN DATA SHAPE:", Training_data.shape)
print("FINAL y_train SHAPE:", y_train.shape)



FINAL TRAIN DATA SHAPE: (5634, 20)
FINAL y_train SHAPE: (5634,)


In [ ]:
# Random over sampling
from imblearn.over_sampling import SMOTE

ros = SMOTE(sampling_strategy=1.0, random_state=42)
Training_data_resample, y_train_resample = ros.fit_resample(Training_data, y_train)

print("AFTER OVERSAMPLING:", Training_data_resample.shape, y_train_resample.shape)

AFTER OVERSAMPLING: (8236, 20) (8236,)


In [ ]:
Training_data_resample.sample(10)

,SeniorCitizen,tenure,MonthlyCharges,T_MODE,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_OD,MultipleLines_OD,InternetService_OD,OnlineSecurity_OD,OnlineBackup_OD,DeviceProtection_OD,TechSupport_OD,StreamingTV_OD,StreamingMovies_OD,Contract_OD,PaperlessBilling_OD,PaymentMethod_OD,Sim_OD
945,1,35,99.050000,3554.600000,0.000000,0.0,0.000000,1.0,2.000000,1.0,2.0,0.000000,0.0,0.0,2.000000,2.000000,0.0,1.0,2.000000,2.0
5150,1,10,29.650000,291.400000,0.000000,0.0,0.000000,0.0,1.000000,0.0,0.0,0.000000,0.0,2.0,0.000000,0.000000,0.0,1.0,2.000000,0.0
3531,0,7,19.400000,168.650000,1.000000,0.0,0.000000,1.0,0.000000,2.0,1.0,1.000000,1.0,1.0,1.000000,1.000000,0.0,1.0,3.000000,1.0
6869,0,15,86.432726,1278.263883,0.761038,0.0,0.238962,1.0,1.522076,1.0,0.0,0.477924,0.0,0.0,1.522076,0.477924,0.0,1.0,1.522076,2.0
2801,0,72,115.800000,8476.500000,0.000000,1.0,1.000000,1.0,2.000000,1.0,2.0,2.000000,2.0,2.0,2.000000,2.000000,2.0,1.0,0.000000,2.0
7215,1,3,79.013071,260.049646,0.817128,0.0,0.000000,1.0,0.365744,1.0,0.0,0.000000,0.0,0.0,1.634256,0.000000,0.0,1.0,1.817128,2.0
304,0,46,90.950000,4236.600000,0.000000,1.0,1.000000,1.0,0.000000,1.0,0.0,0.000000,2.0,2.0,0.000000,2.000000,1.0,0.0,0.000000,2.0
6723,1,9,82.915174,790.840959,1.000000,0.0,0.000000,1.0,1.593554,1.0,0.0,0.000000,0.0,0.0,0.000000,2.000000,0.0,1.0,2.000000,2.0
3020,0,1,74.500000,74.500000,0.000000,0.0,0.000000,1.0,2.000000,1.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,1.0,2.000000,2.0
4382,0,59,51.700000,3005.800000,1.000000,0.0,0.000000,0.0,1.000000,0.0,0.0,0.000000,0.0,2.0,2.000000,2.000000,0.0,1.0,2.000000,0.0


In [ ]:
# feature Scaling
# Z-score

import sklearn

In [ ]:
from sklearn.preprocessing import StandardScaler
obj = StandardScaler()
obj.fit(Training_data_resample)

StandardScaler()

In [ ]:
Training_data_sc = obj.transform(Training_data_resample)

In [ ]:
Testing_data_sc = obj.transform(Testing_data)

In [ ]:
Testing_data_sc

array([[ 2.50897415,  0.31044069,  1.49449738, ...,  0.76910275,
        -1.68038965,  0.7587514 ],
       [-0.39856927, -0.61084944, -1.30663975, ...,  0.76910275,
        -1.68038965, -1.38131483],
       [-0.39856927,  1.39923811, -1.51160101, ...,  0.76910275,
        -0.65597057,  1.82878452],
       ...,
       [-0.39856927,  0.89671622,  0.95494126, ...,  0.76910275,
        -1.68038965,  0.7587514 ],
       [-0.39856927,  0.14293339,  0.76224265, ...,  0.76910275,
         0.36844851,  0.7587514 ],
       [-0.39856927, -1.11337133,  0.53450792, ...,  0.76910275,
         1.39286759,  0.7587514 ]])

In [ ]:
from sklearn.preprocessing import StandardScaler

obje = StandardScaler()

obje.fit(Training_data_resample)

StandardScaler()

In [ ]:
Training_data_Zscore = obje.transform(Training_data_resample)

In [ ]:
Training_data_Zscore

array([[-0.39856927,  1.39923811, -0.29234432, ..., -1.39584233,
         0.36844851, -1.38131483],
       [-0.39856927, -0.1920812 ,  0.02122888, ...,  0.76910275,
        -1.68038965, -1.38131483],
       [-0.39856927, -1.02961768, -0.82489322, ..., -1.39584233,
         0.36844851, -1.38131483],
       ...,
       [-0.39856927, -1.11337133, -1.70006674, ..., -1.39584233,
         1.39286759, -0.31128171],
       [-0.39856927, -1.11337133,  0.27468519, ..., -0.87660391,
        -0.12294293,  0.7587514 ],
       [-0.39856927, -1.11337133, -0.80777895, ...,  0.76910275,
         1.39286759, -1.38131483]])

In [ ]:
Testing_data_Zscore = obje.transform(Testing_data)

In [ ]:
Testing_data_Zscore

array([[ 2.50897415,  0.31044069,  1.49449738, ...,  0.76910275,
        -1.68038965,  0.7587514 ],
       [-0.39856927, -0.61084944, -1.30663975, ...,  0.76910275,
        -1.68038965, -1.38131483],
       [-0.39856927,  1.39923811, -1.51160101, ...,  0.76910275,
        -0.65597057,  1.82878452],
       ...,
       [-0.39856927,  0.89671622,  0.95494126, ...,  0.76910275,
        -1.68038965,  0.7587514 ],
       [-0.39856927,  0.14293339,  0.76224265, ...,  0.76910275,
         0.36844851,  0.7587514 ],
       [-0.39856927, -1.11337133,  0.53450792, ...,  0.76910275,
         1.39286759,  0.7587514 ]])

In [ ]:
import sklearn

from sklearn.model_selection import GridSearchCV,cross_validate

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score,confusion_matrix

In [ ]:
def lr(X_train,y_train,X_test,y_test):
    global lr_reg
    lr_reg = LogisticRegression()
    lr_reg.fit(X_train,y_train)
    predictions = lr_reg.predict(X_test)
    print(confusion_matrix(y_test,predictions))
    print(accuracy_score(y_test,predictions))
    print(classification_report(y_test,predictions))
    global lr_predictions
    lr_predictions = lr_reg.predict(X_test)

lr(Training_data_Zscore,y_train_resample,Testing_data_Zscore,y_test)

InvalidParameterError: The 'penalty' parameter of LogisticRegression must be a str among {'elasticnet', 'l1', 'l2'} or None. Got {'C': 10, 'class_weight': 'balanced', 'max_iter': 100, 'penalty': 'l2', 'solver': 'liblinear'} instead.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
parameters_list = [
    # ----- L2 -----
    {
        'penalty': ['l2'],
        'solver': ['lbfgs', 'sag', 'saga', 'newton-cg', 'liblinear'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'max_iter': [100, 200, 500],
        'class_weight': [None, 'balanced'],
    },

    # ----- L1 -----
    {
        'penalty': ['l1'],
        'solver': ['liblinear', 'saga'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'max_iter': [100, 200, 500],
        'class_weight': [None, 'balanced'],
    },

    # ----- ELASTICNET -----
    {
        'penalty': ['elasticnet'],
        'solver': ['saga'],  # ONLY saga supports elasticnet
        'l1_ratio': [0.0, 0.2, 0.5, 0.8, 1.0],
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'max_iter': [100, 200, 500],
        'class_weight': [None, 'balanced'],
    }
]

grid_reg = GridSearchCV(estimator=LogisticRegression(),param_grid=parameters_list,scoring='accuracy',cv=10)
grid_result = grid_reg.fit(Training_data_Zscore, y_train_resample)
print(grid_result)
print(grid_result.best_params_)
print(grid_result.best_score_)

GridSearchCV(cv=10, estimator=LogisticRegression(),
             param_grid=[{'C': [0.001, 0.01, 0.1, 1, 10, 100],
                          'class_weight': [None, 'balanced'],
                          'max_iter': [100, 200, 500], 'penalty': ['l2'],
                          'solver': ['lbfgs', 'sag', 'saga', 'newton-cg',
                                     'liblinear']},
                         {'C': [0.001, 0.01, 0.1, 1, 10, 100],
                          'class_weight': [None, 'balanced'],
                          'max_iter': [100, 200, 500], 'penalty': ['l1'],
                          'solver': ['liblinear', 'saga']},
                         {'C': [0.001, 0.01, 0.1, 1, 10, 100],
                          'class_weight': [None, 'balanced'],
                          'l1_ratio': [0.0, 0.2, 0.5, 0.8, 1.0],
                          'max_iter': [100, 200, 500],
                          'penalty': ['elasticnet'], 'solver': ['saga']}],
             scoring='accuracy')


In [ ]:
print(grid_result.best_params_)


{'C': 1, 'class_weight': None, 'max_iter': 100, 'penalty': 'l2', 'solver': 'sag'}


In [ ]:
print(grid_result.best_score_)

0.7751412662647902
